In [18]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal

In [19]:
class QuadState(TypedDict, total=False):
    a: float
    b: float
    c: float
    discriminant: float
    equation: str
    result: str

In [20]:
def show_equation(state: QuadState):
    state['equation'] = f"{state['a']}x^2 + {state['b']}x + {state['c']} = 0"
    return state


def calculate_discriminant(state: QuadState):
    if state['a'] == 0:
        raise ValueError("Coefficient 'a' must be non-zero for a quadratic equation.")

    state['discriminant'] = state['b'] ** 2 - 4 * state['a'] * state['c']
    return state


def route_by_discriminant(state: QuadState) -> Literal['real_roots', 'no_real_roots']:
    if state['discriminant'] < 0:
        return 'no_real_roots'
    return 'real_roots'


def real_roots(state: QuadState):
    discriminant = state['discriminant']
    a = state['a']
    b = state['b']

    if discriminant > 0:
        root1 = (-b + discriminant ** 0.5) / (2 * a)
        root2 = (-b - discriminant ** 0.5) / (2 * a)
        state['result'] = f"Two distinct real roots: {root1} and {root2}"
    else:
        root = -b / (2 * a)
        state['result'] = f"One real root: {root}"
    return state


def no_real_roots(state: QuadState):
    state['result'] = "No real roots"
    return state

In [21]:
graph = StateGraph(QuadState)

graph.add_node('show_equation', show_equation)
graph.add_node('calculate_discriminant', calculate_discriminant)
graph.add_node('real_roots', real_roots)
graph.add_node('no_real_roots', no_real_roots)

graph.add_edge(START, 'show_equation')
graph.add_edge('show_equation', 'calculate_discriminant')
graph.add_conditional_edges(
    'calculate_discriminant',
    route_by_discriminant,
    {
        'real_roots': 'real_roots',
        'no_real_roots': 'no_real_roots',
    },
)
graph.add_edge('real_roots', END)
graph.add_edge('no_real_roots', END)

workflow = graph.compile()

In [22]:
initial_state={
  'a': 4,
  'b': -5,
  'c': -4
}
workflow.invoke(initial_state)

{'a': 4,
 'b': -5,
 'c': -4,
 'discriminant': 89,
 'equation': '4x^2 + -5x + -4 = 0',
 'result': 'Two distinct real roots: 1.8042476415070754 and -0.5542476415070754'}